# 30_meta_invertedpendulum_rl2.ipynb
## Revize 3A (InvertedPendulum-RL²): Model-Free Meta-RL (RNN policy + PPO) – Continuous

**Amaç:** z_t'nin (örtük) çıkarımı, epizod-içi adaptasyon, senaryo bazlı performans ve render.

**Ortam:** `InvertedPendulumMeta-v1` — Gymnasium MuJoCo `InvertedPendulum-v4` tabanlı; her epizotta `{gravity g, actuator gain_scale}` yeniden örneklenir.

**Yöntem:** Model-free RL² (LSTM policy) + PPO (sb3-contrib `RecurrentPPO`).

**Girdi genişletme:** `[obs_t, a_{t-1}, r_{t-1}, done_{t-1}]`.

**Metrikler:**
- Adaptation curve (aynı task içinde ardışık epizot ödülleri)
- Regret-benzeri ölçüm: `Regret_k = R* - R_k`
- Senaryo karşılaştırmaları: nominal / düşük-g / yüksek-g / zayıf motor / güçlü motor

**Render:** `evaluate_adaptation(..., render_mode="human"|"rgb_array"|None)`

> Gereksinimler: `gymnasium[mujoco]`, `stable-baselines3>=2.3`, `sb3-contrib>=2.3`, `numpy`, `matplotlib`, (opsiyonel) `scikit-learn`.


In [ ]:
# Kurulum (gerekiyorsa açın)
# !pip install -q gymnasium==0.29.1 gymnasium[mujoco]==0.29.1
# !pip install -q stable-baselines3==2.3.2 sb3-contrib==2.3.2
# !pip install -q numpy matplotlib scikit-learn

## 1) Meta-Ortam: InvertedPendulumMeta-v1
- `InvertedPendulum-v4` üzerinde çalışır.
- Her `reset()` çağrısında **task** parametreleri örneklenir: `g ∈ [7.5, 12.0]` ve `gain_scale ∈ [0.7, 1.3]`.
- MuJoCo'da yerçekimi `env.model.opt.gravity[2]` üzerinden ayarlanır (z-ekseni negatif).
- Aktüatör kazancı `env.model.actuator_gainprm[0,0]` ölçeklenir (tek aktüatör varsayımı `InvertedPendulum` için geçerlidir).
- RL² için gözleme `[prev_action(1), prev_reward(1), prev_done(1)]` eklenir ⇒ toplam obs boyutu `obs_dim + 3`.
- `info['task']` içine gerçek task parametreleri yazılır.

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium.spaces import Box
from typing import Optional, Dict, Any, Tuple

class InvertedPendulumMetaEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 50}

    def __init__(self,
                 render_mode: Optional[str] = None,
                 g_range: Tuple[float,float] = (7.5, 12.0),
                 gain_range: Tuple[float,float] = (0.7, 1.3),
                 max_episode_steps: int = 1000,
                 seed: Optional[int] = None):
        super().__init__()
        self.render_mode = render_mode
        self.base_env = gym.make("InvertedPendulum-v4", render_mode=render_mode)
        self._elapsed_steps = 0
        self.max_episode_steps = max_episode_steps
        self.g_range = g_range
        self.gain_range = gain_range
        self.task = None

        self.orig_observation_space: Box = self.base_env.observation_space  # (4,)
        self.orig_action_space: Box = self.base_env.action_space            # (1,)

        low = np.concatenate([
            self.orig_observation_space.low,
            np.array([-np.inf, -np.inf, -np.inf], dtype=np.float32)
        ]).astype(np.float32)
        high = np.concatenate([
            self.orig_observation_space.high,
            np.array([np.inf, np.inf, np.inf], dtype=np.float32)
        ]).astype(np.float32)
        self.observation_space = Box(low, high, dtype=np.float32)
        self.action_space = self.orig_action_space

        self.prev_a = np.zeros(self.action_space.shape, dtype=np.float32)
        self.prev_r = 0.0
        self.prev_done = 0.0
        self.np_random = np.random.default_rng(seed)

    def _sample_task(self) -> Dict[str, float]:
        g = float(self.np_random.uniform(*self.g_range))
        gain_scale = float(self.np_random.uniform(*self.gain_range))
        return {"g": g, "gain_scale": gain_scale}

    def _apply_task_to_env(self, task: Dict[str, float]):
        env = self.base_env.unwrapped
        env.model.opt.gravity[2] = -abs(task["g"])        # örn. -9.81
        if env.model.actuator_gainprm.shape[0] >= 1:       # tek aktüatör
            env.model.actuator_gainprm[0, 0] = task["gain_scale"]

    def _aug_obs(self, obs: np.ndarray) -> np.ndarray:
        return np.concatenate([
            obs.astype(np.float32),
            self.prev_a.astype(np.float32).reshape(-1),
            np.array([self.prev_r, self.prev_done], dtype=np.float32)
        ]).astype(np.float32)

    def reset(self, *, seed: Optional[int] = None, options: Optional[Dict[str, Any]] = None):
        if seed is not None:
            self.base_env.reset(seed=seed)
        self._elapsed_steps = 0
        if options is not None and options.get("task") is not None:
            self.task = options["task"]
        else:
            self.task = self._sample_task()
        self._apply_task_to_env(self.task)
        obs, info = self.base_env.reset()
        self.prev_a[:] = 0.0
        self.prev_r = 0.0
        self.prev_done = 0.0
        aug = self._aug_obs(obs)
        info = info or {}
        info.update({"task": self.task})
        return aug, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.base_env.step(action)
        self._elapsed_steps += 1
        done = terminated or truncated or (self._elapsed_steps >= self.max_episode_steps)
        self.prev_a = np.array(action, dtype=np.float32)
        self.prev_r = float(reward)
        self.prev_done = float(done)
        aug = self._aug_obs(obs)
        info = info or {}
        info.update({"task": self.task})
        return aug, reward, terminated, truncated, info

    def render(self):
        return self.base_env.render()

    def close(self):
        self.base_env.close()

# Opsiyonel hızlı test
if __name__ == "__main__":
    try:
        env = InvertedPendulumMetaEnv()
        obs, info = env.reset()
        total = 0
        for _ in range(50):
            a = env.action_space.sample()
            obs, r, term, trunc, inf = env.step(a)
            total += r
            if term or trunc:
                break
        env.close()
        print("Smoke test reward:", total)
    except Exception as e:
        print("Smoke test skipped:", e)

## 2) Task Sampler ve Senaryolar
- Eğitim/Değerlendirme aralıkları ve sabit senaryolar (nominal, low-g, high-g, weak/strong motor).

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class TaskRanges:
    g_range: tuple = (7.5, 12.0)
    gain_range: tuple = (0.7, 1.3)

class TaskSampler:
    def __init__(self, ranges: TaskRanges, seed: int = 42):
        self.ranges = ranges
        self.rng = np.random.default_rng(seed)
    def sample(self, n: int) -> List[dict]:
        tasks = []
        for _ in range(n):
            g = float(self.rng.uniform(*self.ranges.g_range))
            gain_scale = float(self.rng.uniform(*self.ranges.gain_range))
            tasks.append({"g": g, "gain_scale": gain_scale})
        return tasks

train_ranges = TaskRanges()
eval_ranges = TaskRanges(g_range=(7.8, 11.5), gain_range=(0.8, 1.2))
train_sampler = TaskSampler(train_ranges, seed=123)
eval_sampler = TaskSampler(eval_ranges, seed=999)

SCENARIOS = {
    "nominal": {"g": 9.81, "gain_scale": 1.0},
    "low_g":   {"g": 7.8,  "gain_scale": 1.0},
    "high_g":  {"g": 11.5, "gain_scale": 1.0},
    "weak":    {"g": 9.81, "gain_scale": 0.8},
    "strong":  {"g": 9.81, "gain_scale": 1.2}
}


## 3) RL² Core — **DÜZELTİLMİŞ VECENV** (DummyVecEnv)
- Önceki hata: `make_vec_env(..., env_fn=...)` parametresi bazı sürümlerde yok ⇒ **DummyVecEnv(callables)** kullanıyoruz.
- Eğitim çıktıları: `runs/30_invertedpendulum_rl2/`.
- Değerlendirme tek ortamda ardışık epizotlarla yapılır; `render_mode` opsiyoneldir.

In [ ]:
import os, time
import matplotlib.pyplot as plt
from typing import Callable, Optional
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from sb3_contrib import RecurrentPPO
from sb3_contrib.ppo_recurrent.policies import MlpLstmPolicy

RUNS_DIR = os.path.join("runs", "30_invertedpendulum_rl2")
os.makedirs(RUNS_DIR, exist_ok=True)

def make_ip_meta_env_callable(task_ranges: TaskRanges, render_mode=None) -> Callable[[], gym.Env]:
    def _thunk():
        env = InvertedPendulumMetaEnv(
            render_mode=render_mode,
            g_range=task_ranges.g_range,
            gain_range=task_ranges.gain_range,
            max_episode_steps=1000,
        )
        return env
    return _thunk

def build_vec_env(n_envs: int = 8, for_eval: bool = False, render_mode=None) -> VecMonitor:
    sampler = eval_sampler if for_eval else train_sampler
    env_fns = [make_ip_meta_env_callable(sampler.ranges, render_mode=render_mode) for _ in range(n_envs)]
    vec = DummyVecEnv(env_fns)
    vec = VecMonitor(vec)
    return vec

def train_recurrent_ppo(total_timesteps=400_000,
                        n_envs=8,
                        learning_rate=3e-4,
                        gamma=0.99,
                        ent_coef=0.0,
                        gae_lambda=0.95,
                        n_steps=128,
                        batch_size=256,
                        n_epochs=5,
                        clip_range=0.2,
                        lstm_hidden_size=128,
                        log_dir=RUNS_DIR):
    env = build_vec_env(n_envs=n_envs, for_eval=False)
    policy_kwargs = dict(
        lstm_hidden_size=lstm_hidden_size,
        ortho_init=False,
        net_arch=[128, 128],
        enable_critic_lstm=True,
    )
    model = RecurrentPPO(
        policy=MlpLstmPolicy,
        env=env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        gamma=gamma,
        gae_lambda=gae_lambda,
        ent_coef=ent_coef,
        clip_range=clip_range,
        verbose=1,
        tensorboard_log=log_dir,
        policy_kwargs=policy_kwargs,
        seed=42,
    )
    model.learn(total_timesteps=total_timesteps, progress_bar=True)
    save_path = os.path.join(log_dir, "recurrent_ppo_invertedpendulum_meta.zip")
    model.save(save_path)
    env.close()
    print("Model saved to:", save_path)
    return save_path

def evaluate_adaptation(model_path: str,
                        n_tasks: int = 10,
                        episodes_per_task: int = 5,
                        max_steps_per_ep: int = 1000,
                        render_mode: Optional[str] = None,
                        scenario: Optional[str] = None):
    """Aynı task üstünde ardışık epizotlarla adaptasyon. Senaryo adı verilirse sabit task kullanır."""
    env = InvertedPendulumMetaEnv(render_mode=render_mode)
    model = RecurrentPPO.load(model_path)

    ep_rewards_across_tasks = []  # [n_tasks, episodes_per_task]
    regret_like = []
    task_params = []
    frames_for_display = []  # rgb_array modunda örnek kareler

    for t in range(n_tasks):
        task = SCENARIOS[scenario] if (scenario is not None and scenario in SCENARIOS) else eval_sampler.sample(1)[0]
        task_params.append(task)

        # Basit R* kestirimi: aynı task'te 3 kısa deterministik denemeden en iyisi
        ref_scores = []
        for _ in range(3):
            obs, info = env.reset(options={"task": task})
            done = False
            lstm_states = None
            ep_r = 0.0
            for _ in range(max_steps_per_ep):
                action, lstm_states = model.predict(obs, state=lstm_states, episode_start=np.array([done]), deterministic=True)
                obs, r, term, trunc, inf = env.step(action)
                done = (term or trunc)
                ep_r += r
                if done:
                    break
            ref_scores.append(ep_r)
        R_star = float(np.max(ref_scores))

        # Adaptasyon: ardışık epizotlar (stochastic policy)
        task_ep_rewards = []
        lstm_states = None
        for ep in range(episodes_per_task):
            obs, info = env.reset(options={"task": task})
            done = False
            ep_r = 0.0
            steps = 0
            episode_start = True
            while True:
                action, lstm_states = model.predict(
                    obs,
                    state=lstm_states,
                    episode_start=np.array([episode_start]),
                    deterministic=False,
                )
                episode_start = False
                obs, r, term, trunc, inf = env.step(action)
                done = (term or trunc)
                ep_r += r
                steps += 1
                # rgb_array render: birkaç kare sakla
                if render_mode == "rgb_array" and t == 0 and ep == 0 and steps % 200 == 0:
                    frame = env.render()
                    if frame is not None:
                        frames_for_display.append(frame)
                if done or steps >= max_steps_per_ep:
                    break
            task_ep_rewards.append(ep_r)
        ep_rewards_across_tasks.append(task_ep_rewards)
        regret_like.append([R_star - r for r in task_ep_rewards])

    env.close()
    return (np.array(ep_rewards_across_tasks),
            np.array(regret_like),
            task_params,
            frames_for_display)

def plot_adaptation_curves(ep_rewards: np.ndarray, title: str = "Adaptation Curve"):
    mean = ep_rewards.mean(axis=0)
    std = ep_rewards.std(axis=0)
    xs = np.arange(1, ep_rewards.shape[1]+1)
    plt.figure(figsize=(6,4))
    plt.plot(xs, mean, label="Ortalama ödül")
    plt.fill_between(xs, mean-std, mean+std, alpha=0.2)
    plt.xlabel("Aynı task içinde epizot #")
    plt.ylabel("Ödül")
    plt.title(title)
    plt.grid(True)
    plt.legend(); plt.show()

def plot_regret(regret_like: np.ndarray, title: str = "Regret-benzeri Ölçüm"):
    mean = regret_like.mean(axis=0)
    std = regret_like.std(axis=0)
    xs = np.arange(1, regret_like.shape[1]+1)
    plt.figure(figsize=(6,4))
    plt.plot(xs, mean, label="Ortalama regret")
    plt.fill_between(xs, mean-std, mean+std, alpha=0.2)
    plt.xlabel("Aynı task içinde epizot #")
    plt.ylabel("Regret (R* - R_k)")
    plt.title(title)
    plt.grid(True)
    plt.legend(); plt.show()

def display_rgb_frames(frames):
    if len(frames) == 0:
        print("Gösterilecek kare yok (render_mode='rgb_array' ile toplayın).")
        return
    n = len(frames)
    cols = min(3, n)
    rows = int(np.ceil(n/cols))
    plt.figure(figsize=(4*cols, 3*rows))
    for i, f in enumerate(frames):
        plt.subplot(rows, cols, i+1)
        plt.imshow(f)
        plt.axis('off')
    plt.suptitle("Render Frames (rgb_array)"); plt.show()


## 4) Eğitim
- Varsayılan: `total_timesteps=400k`, `n_envs=8`. Hızlı denemeler için azaltabilirsiniz.
- Çıktılar: `runs/30_invertedpendulum_rl2/` altında kayıt edilir.

In [ ]:
MODEL_PATH = train_recurrent_ppo(
    total_timesteps=400_000,
    n_envs=8,
    learning_rate=3e-4,
    n_steps=128,
    batch_size=256,
    n_epochs=5,
    lstm_hidden_size=128,
    log_dir=RUNS_DIR,
)
MODEL_PATH

## 5) Değerlendirme: Adaptasyon, Regret ve Senaryolar
- **Rastgele held-out tasklar** ve **sabit senaryolar** için değerlendirme.
- `render_mode`: `None` (varsayılan), `"human"` (canlı pencere), `"rgb_array"` (inline kareler).

In [ ]:
# (A) Held-out random tasks, no render
ep_rewards, regret_like, tasks, _ = evaluate_adaptation(
    MODEL_PATH,
    n_tasks=10,
    episodes_per_task=5,
    max_steps_per_ep=1000,
    render_mode=None,
)
print("Held-out ep_rewards shape:", ep_rewards.shape)
plot_adaptation_curves(ep_rewards, title="InvertedPendulumMeta RL² – Adaptation (Held-out)")
plot_regret(regret_like, title="InvertedPendulumMeta RL² – Regret (Held-out)")

# (B) Sabit senaryolar: nominal / low_g / high_g / weak / strong
scenario_results = {}
for sc in ["nominal", "low_g", "high_g", "weak", "strong"]:
    er, rg, tp, _ = evaluate_adaptation(
        MODEL_PATH,
        n_tasks=6,  # aynı senaryonun farklı epizotları
        episodes_per_task=5,
        max_steps_per_ep=1000,
        render_mode=None,
        scenario=sc,
    )
    scenario_results[sc] = (er, rg)

# Senaryolar arası ortalama ödül karşılaştırması (1. ve 5. epizot)
plt.figure(figsize=(6,4))
x = np.arange(len(scenario_results))
first_ep = [scenario_results[k][0][:,0].mean() for k in scenario_results]
last_ep  = [scenario_results[k][0][:, -1].mean() for k in scenario_results]
width=0.35
plt.bar(x - width/2, first_ep, width, label="Ep1")
plt.bar(x + width/2, last_ep, width, label="Ep5")
plt.xticks(x, list(scenario_results.keys()))
plt.ylabel("Ortalama ödül")
plt.title("Senaryolar Arası Adaptasyon Karşılaştırması")
plt.legend(); plt.grid(True, axis='y', alpha=0.3); plt.show()

# (C) Render örneği: rgb_array ile birkaç kare göster
er, rg, tp, frames = evaluate_adaptation(
    MODEL_PATH,
    n_tasks=1,
    episodes_per_task=2,
    max_steps_per_ep=500,
    render_mode="rgb_array",
    scenario="nominal",
)
display_rgb_frames(frames)


## 6) (Opsiyonel) t-SNE: LSTM durumlarının proxy görselleştirmesi
- Burada örnek olarak sadece state normlarından sentetik bir örnek gösteriliyor.  
- Pratikte rollout sırasında (h,c) vektörlerini toplayıp gömü öğrenimi analiz edilebilir.

In [ ]:
try:
    from sklearn.manifold import TSNE
    rng = np.random.default_rng(0)
    fake_hidden_norms = rng.normal(0, 1, size=(2000, 1))
    emb = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=0).fit_transform(fake_hidden_norms)
    plt.figure(figsize=(5,5))
    plt.scatter(emb[:,0], emb[:,1], s=2, alpha=0.6)
    plt.title("t-SNE of LSTM state norms (proxy demo)")
    plt.axis('off'); plt.show()
except Exception as e:
    print("t-SNE atlandı:", e)


## 7) Notlar ve İyileştirmeler
- **R\*** referansı akademik raporlama için ayrı bir task-özel PPO ile ölçülmelidir (burada hızlı bir kestirim kullanıldı).
- Girdi genişletmeye *reward normalization* ve *action clipping/normalization* eklenebilir.
- LSTM boyutları ve PPO hiperparametreleri (n_steps, batch_size, lr) senaryolara göre ayarlanmalıdır.
- **Render**: `render_mode="human"` ile canlı pencere; `"rgb_array"` ile notebook içine kare düşer.
- **Genelleme**: eval aralıklarını daraltıp uç senaryoları (ör. çok yüksek g) dış dağıtım testi için ayrı tutabilirsiniz.
- **Kayıt**: TensorBoard loglarını `runs/30_invertedpendulum_rl2/` altında inceleyin.
